# Final Project Solution Exemplar — Hospital Readmission Risk (Scenario 06)

**[Your Name] — ISM6251 Spring 2026**

---

This notebook is an *exemplar* — it shows the technical execution and writing depth expected for full marks. See `solution-exemplar.md` in this folder for the full written Parts 1, 4.2, and 5 that accompany this notebook. Your actual submission should combine notebook code with equivalent depth of written reasoning.

**Scenario 06 at a glance:** MercyFirst Health System wants to flag high-risk patients at discharge for a post-discharge care bundle. FP cost = $500 (unnecessary bundle); FN cost = $25,000 (readmission + CMS penalty exposure). FN:FP ratio = 50:1.

## Setup — Path Helper and Imports

This cell locates the scenario data regardless of where you run the notebook from. Copy this pattern to your own submission.

In [ ]:
import os
from pathlib import Path

SCENARIO = '06-hospital-readmission'

# Try common relative paths so the notebook runs from anywhere in the repo
candidates = [
    Path(f'cases/{SCENARIO}'),
    Path(f'../cases/{SCENARIO}'),
    Path(f'../../cases/{SCENARIO}'),
    Path(f'FinalProject/cases/{SCENARIO}'),
]
DATA_DIR = next((p for p in candidates if p.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        f"Couldn't find cases/{SCENARIO}. Run this notebook from the FinalProject folder, "
        f"or update `candidates` with the correct path to your scenario data."
    )
print(f'Data directory: {DATA_DIR.resolve()}')

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               VotingClassifier, StackingClassifier)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
from sklearn.metrics import (f1_score, precision_score, recall_score, roc_auc_score,
                             average_precision_score, confusion_matrix, classification_report,
                             precision_recall_curve)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Costs from scenario.md
FP_COST = 500
FN_COST = 25000
RANDOM_STATE = 42

# Set random seed everywhere for reproducibility
np.random.seed(RANDOM_STATE)

## Part 1: Business Understanding

**See `solution-exemplar.md` for the full written Parts 1.1–1.4.** Below is the executable summary.

### 1.3 Null Baseline Identification (computed)

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
# NOTE: We deliberately do NOT load holdout.csv here — that's for the instructor.

X_train, y_train = train.drop('target', axis=1), train['target']
X_test, y_test = test.drop('target', axis=1), test['target']

n_pos_test = int(y_test.sum())
n_neg_test = len(y_test) - n_pos_test
do_nothing_cost = n_pos_test * FN_COST        # predict all negative → miss every positive
flag_all_cost   = n_neg_test * FP_COST        # predict all positive → false alarm on every negative
null_cost = min(do_nothing_cost, flag_all_cost)
null_strategy = 'flag everything' if flag_all_cost < do_nothing_cost else 'do nothing'

print(f'Test set: {n_pos_test} positives, {n_neg_test} negatives ({y_test.mean():.1%} positive rate)')
print(f'\n"Do nothing"     null cost: ${do_nothing_cost:>12,}  ({n_pos_test} missed readmissions × $25K)')
print(f'"Flag everything" null cost: ${flag_all_cost:>12,}  ({n_neg_test} unnecessary bundles × $500)')
print(f'\nBest null baseline: "{null_strategy}" at ${null_cost:,}')
print(f'My model must beat this to be worth deploying.')

## Part 2: Data Exploration

### 2.1 Class distribution and feature overview

In [ ]:
print(f'Training set: {X_train.shape}')
print(f'Test set:     {X_test.shape}')
print(f'Target rate (train): {y_train.mean():.1%}')
print(f'Target rate (test):  {y_test.mean():.1%}')
print(f'Missing values: {train.isnull().sum().sum()}')
print()
train.describe().round(2)

### 2.3 Preprocessing Decisions

**Tree-based models** (RF, GB, XGBoost, LightGBM) are scale-invariant — I don't need to scale features for them, and scaling them would only add noise. **Logistic Regression** does need scaling because its regularization term penalizes large coefficients, which is scale-dependent.

**My solution:** scaling lives *inside* the LogReg pipeline only. This ensures the scaler is refit per cross-validation fold (no leakage), and tree models see raw features (correct behavior). See Part 5.2 for the full leakage check.

## Part 3: Model Development

### 3.1 Baseline + 3.2 Model Exploration (6 models, including ensemble)

I'm training 6 model types: LogReg (linear), Decision Tree (single tree), Random Forest (bagging), Gradient Boosting, XGBoost, LightGBM (all boosting), plus a VotingClassifier ensemble.

All use `class_weight='balanced'` or `scale_pos_weight` to handle the 18% positive rate. Cross-validation with `StratifiedKFold(n_splits=5)` throughout.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

models = {
    'LogReg': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced'))
    ]),
    'DecisionTree': DecisionTreeClassifier(max_depth=8, random_state=RANDOM_STATE, class_weight='balanced'),
    'RandomForest': RandomForestClassifier(n_estimators=200, max_depth=15, random_state=RANDOM_STATE,
                                            n_jobs=-1, class_weight='balanced'),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=500, learning_rate=0.05, max_depth=4,
                                                    n_iter_no_change=15, validation_fraction=0.1,
                                                    random_state=RANDOM_STATE),
    'XGBoost': XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=5, subsample=0.8,
                              colsample_bytree=0.8, scale_pos_weight=pos_weight,
                              random_state=RANDOM_STATE, n_jobs=-1, eval_metric='logloss',
                              early_stopping_rounds=30),
    'LightGBM': LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=31, subsample=0.8,
                                colsample_bytree=0.8, is_unbalance=True,
                                random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
}

In [ ]:
results = {}
for name, model in models.items():
    if name == 'XGBoost':
        model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    elif name == 'LightGBM':
        model.fit(X_train, y_train, eval_set=[(X_test, y_test)])
    else:
        model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    results[name] = {
        'test_auc': round(roc_auc_score(y_test, y_prob), 4),
        'test_ap': round(average_precision_score(y_test, y_prob), 4),
        'test_f1': round(f1_score(y_test, y_pred), 4),
    }
pd.DataFrame(results).T.sort_values('test_ap', ascending=False)

### 3.2.1 VotingClassifier (the ensemble requirement)

*Per Part 3 requirements, I need to try an ensemble combination. I'm expecting this not to help much — my best three models (GB, XGBoost, LightGBM) all share similar tree-based assumptions, so a voting classifier won't add genuine diversity. I'm showing this explicitly rather than hiding it, because **failing gracefully is part of the reasoning being tested.**

In [ ]:
voting = VotingClassifier(
    estimators=[
        ('lr', Pipeline([('s', StandardScaler()), ('c', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE))])),
        ('rf', RandomForestClassifier(n_estimators=200, max_depth=15, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
        ('gb', GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=RANDOM_STATE)),
    ],
    voting='soft'
)
voting.fit(X_train, y_train)
y_prob = voting.predict_proba(X_test)[:, 1]
results['VotingClassifier'] = {
    'test_auc': round(roc_auc_score(y_test, y_prob), 4),
    'test_ap': round(average_precision_score(y_test, y_prob), 4),
    'test_f1': round(f1_score(y_test, (y_prob >= 0.5).astype(int)), 4),
}
pd.DataFrame(results).T.sort_values('test_ap', ascending=False)

**Finding:** VotingClassifier tied or slightly underperformed my best single model. This is because my three base learners are not genuinely diverse — GB and the other tree models make similar predictions. See Part 5.1 for discussion.

### 3.3 Hyperparameter Tuning (RandomizedSearchCV on top 2)

RandomizedSearchCV is cheaper than GridSearch and gives similar results on moderate parameter spaces. I'm tuning learning_rate, max_depth, subsample, and colsample_bytree for the top two performers.

In [ ]:
from scipy.stats import randint, uniform

top_names = sorted(results.keys(), key=lambda n: -results[n]['test_ap'])[:2]
print(f'Tuning: {top_names}')

param_dist = {
    'max_depth': randint(3, 8),
    'learning_rate': uniform(0.01, 0.2),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
}

tuned = {}
for name in top_names:
    if 'XGBoost' in name:
        base = XGBClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1, eval_metric='logloss')
    elif 'LightGBM' in name:
        base = LGBMClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
    elif 'GradientBoosting' in name:
        base = GradientBoostingClassifier(n_estimators=200, random_state=RANDOM_STATE)
    else:
        continue
    search = RandomizedSearchCV(base, param_dist, n_iter=20, cv=cv,
                                  scoring='average_precision', random_state=RANDOM_STATE, n_jobs=-1)
    search.fit(X_train, y_train)
    tuned[name] = search.best_estimator_
    print(f'  {name}: best CV AP = {search.best_score_:.4f}, best params = {search.best_params_}')

### 3.4 Threshold Optimization

Minimizing total business cost across thresholds. **This is where most of the dollar value comes from.**

In [ ]:
# Pick best tuned model by test AP
best_name = max(tuned, key=lambda n: average_precision_score(y_test, tuned[n].predict_proba(X_test)[:, 1]))
best_model = tuned[best_name]
y_prob_test = best_model.predict_proba(X_test)[:, 1]

thresholds = np.arange(0.01, 1.0, 0.01)
rows = []
for t in thresholds:
    y_pred_t = (y_prob_test >= t).astype(int)
    cm = confusion_matrix(y_test, y_pred_t)
    tn, fp, fn, tp = cm.ravel()
    rows.append({
        'threshold': round(t, 2), 'fp': fp, 'fn': fn, 'tp': tp,
        'precision': tp/(tp+fp) if (tp+fp) > 0 else 0,
        'recall': tp/(tp+fn) if (tp+fn) > 0 else 0,
        'total_cost': fp * FP_COST + fn * FN_COST,
        'flag_rate': (tp + fp) / len(y_test),
    })
thresh_df = pd.DataFrame(rows)
opt = thresh_df.loc[thresh_df['total_cost'].idxmin()]
print(f'Optimal threshold: {opt["threshold"]:.2f}')
print(f'  Precision: {opt["precision"]:.4f}, Recall: {opt["recall"]:.4f}')
print(f'  Flag rate: {opt["flag_rate"]:.1%} of discharges')
print(f'  Total cost: ${opt["total_cost"]:,}')
print(f'  vs null baseline (${null_cost:,}): savings of ${int(null_cost - opt["total_cost"]):,}')

### Cost Curve Visualization

A picture of where the cost minimum lies is worth a long explanation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: cost curve
ax = axes[0]
ax.plot(thresh_df['threshold'], thresh_df['total_cost'] / 1000, 'b-', linewidth=2, label='Model cost')
ax.axhline(null_cost/1000, color='red', linestyle='--', label=f'Null baseline ("{null_strategy}")')
ax.axvline(opt['threshold'], color='green', linestyle=':', label=f'Optimal threshold = {opt["threshold"]:.2f}')
ax.axvline(0.5, color='gray', linestyle=':', alpha=0.5, label='Default 0.5 threshold')
ax.scatter([opt['threshold']], [opt['total_cost']/1000], color='green', s=100, zorder=5)
ax.set_xlabel('Classification Threshold')
ax.set_ylabel('Total Business Cost ($K)')
ax.set_title('Cost vs. Threshold — Model beats null baseline across a wide range')
ax.legend()
ax.grid(alpha=0.3)

# Right: precision/recall at each threshold
ax = axes[1]
ax.plot(thresh_df['threshold'], thresh_df['precision'], 'g-', linewidth=2, label='Precision')
ax.plot(thresh_df['threshold'], thresh_df['recall'], 'orange', linewidth=2, label='Recall')
ax.plot(thresh_df['threshold'], thresh_df['flag_rate'], 'purple', linewidth=2, label='Flag rate')
ax.axvline(opt['threshold'], color='green', linestyle=':', label=f'Optimal = {opt["threshold"]:.2f}')
ax.set_xlabel('Classification Threshold')
ax.set_ylabel('Rate')
ax.set_title('Precision, Recall, and Flag Rate by Threshold')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Part 4: Model Evaluation & Selection

### 4.1 Model Comparison (with CV confidence intervals)

In [ ]:
cv_scores = {}
for name, model in list(models.items()) + [(f'{n} (tuned)', m) for n, m in tuned.items()]:
    if 'XGBoost' in name or 'LightGBM' in name:
        # skip early-stopping models in CV — would require manual CV loop
        continue
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='average_precision', n_jobs=-1)
    cv_scores[name] = {'mean_ap': round(scores.mean(), 4), 'std_ap': round(scores.std(), 4)}
cv_df = pd.DataFrame(cv_scores).T.sort_values('mean_ap', ascending=False)
cv_df

### 4.2 Final Model Selection

**See `solution-exemplar.md` for the full written defense.** The short version:
- Top 3 models tied within 1 CV standard deviation — choosing between them is within the noise
- Chose Gradient Boosting for operational simplicity (no external C++ deps) + capacity constraint fit
- Explicitly acknowledged that LightGBM could be a reasonable runner-up choice for a different stakeholder

### 4.3 Feature Importance — Built-in vs. Permutation

In [ ]:
# Built-in feature importance
feat_names = X_train.columns.tolist()
builtin = pd.Series(best_model.feature_importances_, index=feat_names).sort_values(ascending=False)

# Permutation importance (takes a minute)
perm = permutation_importance(best_model, X_test, y_test, n_repeats=10,
                               random_state=RANDOM_STATE, scoring='average_precision')
perm_df = pd.Series(perm.importances_mean, index=feat_names).sort_values(ascending=False)

# Compare
comp = pd.DataFrame({'built-in': builtin, 'permutation': perm_df}).round(4)
print('Top 10 by permutation importance:')
print(comp.sort_values('permutation', ascending=False).head(10))

### 4.4 Error Analysis

In [ ]:
y_pred_opt = (y_prob_test >= opt['threshold']).astype(int)
errors = X_test.copy()
errors['actual'] = y_test.values
errors['predicted'] = y_pred_opt
errors['prob'] = y_prob_test
errors['error_type'] = np.where((y_pred_opt == 0) & (y_test == 1), 'FN',
                         np.where((y_pred_opt == 1) & (y_test == 0), 'FP', 'correct'))

print('Error breakdown:')
print(errors['error_type'].value_counts())
print('\nFN (missed readmissions) — mean feature values:')
print(errors[errors['error_type'] == 'FN'].drop(columns=['actual', 'predicted', 'error_type', 'prob']).mean().round(2).head(10))

## Part 5: Critical Reflection

**The full Part 5 narrative is in `solution-exemplar.md`.** It covers:
- **5.1 Algorithm Suitability** — model-by-model analysis grounded in algorithm mechanics, not labels
- **5.2 Data Leakage Check** — pipeline walkthrough with an honest admission of an initial SMOTE leakage that was caught and fixed
- **5.3 Counterfactuals** — 4 specific scenarios with first-order and second-order consequences
- **5.4 Lessons Learned** — surprise about how much threshold choice matters relative to model choice

> **Reminder:** Part 5 is written without AI assistance and is presented in a submitted video in Week 12. See `example-responses.md` for the depth expected.

## Generating Holdout Predictions

This is the last step. We load the holdout set, predict, and save.

In [ ]:
holdout = pd.read_csv(DATA_DIR / 'holdout.csv')
X_holdout = holdout.drop('target', axis=1) if 'target' in holdout.columns else holdout
y_prob_holdout = best_model.predict_proba(X_holdout)[:, 1]

submission = pd.DataFrame({
    'id': range(len(holdout)),
    'predicted_probability': y_prob_holdout
})
submission.to_csv('TeamName_holdout_predictions.csv', index=False)
print(f'Saved {len(submission)} holdout predictions.')
print('\nBefore submitting, run:')
print('  python templates/validate_submission.py TeamName_holdout_predictions.csv 06')